# 💳 Credit Card Default Prediction — End-to-End ML Project
> Dataset: UCI Taiwan Credit Card Default | Period: April–September 2005

---
## Project Outline
1. Import Libraries
2. Load Dataset
3. Exploratory Data Analysis (EDA)
4. Distribution Analysis & Skewness Handling
5. Feature Engineering
6. Class Imbalance Treatment (SMOTE)
7. Model Building & Evaluation
8. Model Comparison
9. Save Best Model

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score,
    precision_score, recall_score, f1_score
)
from imblearn.over_sampling import SMOTE
import joblib, json

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 5)
print('Libraries loaded successfully ✅')

## 2. Load Dataset

In [ ]:
# Load dataset — update path if running locally
df = pd.read_csv('data/Credit_Card_Default.csv')
df.rename(columns={'default.payment.next.month': 'default'}, inplace=True)

print(f'Dataset Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
# Data types & missing values
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())

In [ ]:
# Descriptive statistics
df.describe().T

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# --- Target Variable Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = df['default'].value_counts()
labels = ['No Default (0)', 'Default (1)']
colors = ['#2ecc71', '#e74c3c']

axes[0].bar(labels, counts.values, color=colors, edgecolor='black')
axes[0].set_title('Target Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'black'})
axes[1].set_title('Default Rate', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('models/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Class Imbalance Ratio — No Default: {counts[0]}, Default: {counts[1]}')

In [ ]:
# --- Categorical Feature Analysis ---
cat_features = ['SEX', 'EDUCATION', 'MARRIAGE']
cat_labels = {
    'SEX': {1: 'Male', 2: 'Female'},
    'EDUCATION': {1: 'Graduate', 2: 'University', 3: 'High School', 4: 'Others'},
    'MARRIAGE': {1: 'Married', 2: 'Single', 3: 'Others'}
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, cat_features):
    temp = df.groupby([col, 'default']).size().unstack(fill_value=0)
    temp.index = [cat_labels[col].get(i, str(i)) for i in temp.index]
    temp.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'], edgecolor='black')
    ax.set_title(f'{col} vs Default', fontsize=13, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.legend(['No Default', 'Default'])
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('models/categorical_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Age Distribution by Default ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df[df['default'] == 0]['AGE'].hist(ax=axes[0], bins=30, color='#2ecc71', edgecolor='black', alpha=0.8)
df[df['default'] == 1]['AGE'].hist(ax=axes[0], bins=30, color='#e74c3c', edgecolor='black', alpha=0.6)
axes[0].set_title('Age Distribution by Default Status', fontsize=13, fontweight='bold')
axes[0].legend(['No Default', 'Default'])
axes[0].set_xlabel('Age')

sns.boxplot(x='default', y='LIMIT_BAL', data=df, palette=['#2ecc71', '#e74c3c'], ax=axes[1])
axes[1].set_title('Credit Limit vs Default', fontsize=13, fontweight='bold')
axes[1].set_xticklabels(['No Default', 'Default'])

plt.tight_layout()
plt.savefig('models/age_limit_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Correlation Heatmap ---
plt.figure(figsize=(16, 12))
corr = df.drop('ID', axis=1, errors='ignore').corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, annot_kws={'size': 7})
plt.title('Feature Correlation Matrix', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('models/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Bill Amount Trends ---
bill_cols = ['BILL_AMT1','BILL_AMT2','BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6']
pay_cols  = ['PAY_AMT1','PAY_AMT2','PAY_AMT3','PAY_AMT4','PAY_AMT5','PAY_AMT6']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for group, label, color in [(0, 'No Default', '#2ecc71'), (1, 'Default', '#e74c3c')]:
    subset = df[df['default'] == group]
    axes[0].plot(range(1,7), subset[bill_cols].mean().values, marker='o', label=label, color=color)
    axes[1].plot(range(1,7), subset[pay_cols].mean().values,  marker='o', label=label, color=color)

axes[0].set_title('Avg Bill Amount Over 6 Months', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Month (1=Sep, 6=Apr)'); axes[0].legend()
axes[1].set_title('Avg Payment Amount Over 6 Months', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Month (1=Sep, 6=Apr)'); axes[1].legend()

plt.tight_layout()
plt.savefig('models/bill_payment_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Distribution Analysis & Skewness Transformation

In [ ]:
# Drop ID before processing
df.drop(columns=['ID'], inplace=True, errors='ignore')

# Fix anomalous category values
df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
df['MARRIAGE']  = df['MARRIAGE'].replace({0: 3})

numeric_cols = df.select_dtypes(include=np.number).drop(columns=['default']).columns.tolist()

# Skewness before transformation
skewness = df[numeric_cols].skew().sort_values(ascending=False)
print('Skewness of Features (before transformation):')
print(skewness.to_string())

In [ ]:
# Visualize distributions of key numeric columns
key_cols = ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'PAY_AMT1']
fig, axes = plt.subplots(2, 4, figsize=(20, 8))

for i, col in enumerate(key_cols):
    # Before
    axes[0, i].hist(df[col], bins=40, color='#3498db', edgecolor='black', alpha=0.8)
    axes[0, i].set_title(f'{col}\nSkew: {df[col].skew():.2f}', fontsize=11)
    axes[0, i].set_xlabel('Before Transformation')

    # Log1p transformation for skewed
    if df[col].min() >= 0:
        transformed = np.log1p(df[col])
    else:
        transformed = df[col]  # skip negative columns
    axes[1, i].hist(transformed, bins=40, color='#e67e22', edgecolor='black', alpha=0.8)
    axes[1, i].set_title(f'{col}\nSkew after: {transformed.skew():.2f}', fontsize=11)
    axes[1, i].set_xlabel('After Log1p')

plt.suptitle('Distribution: Before vs After Log1p Transformation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('models/distribution_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Apply log1p transformation on highly skewed positive features
skewed_cols = [c for c in numeric_cols if abs(df[c].skew()) > 1 and df[c].min() >= 0]
print(f'Columns with |skew| > 1 (positive): {skewed_cols}')

df_transformed = df.copy()
for col in skewed_cols:
    df_transformed[col] = np.log1p(df_transformed[col])

print('Skewness AFTER transformation:')
print(df_transformed[skewed_cols].skew().sort_values(ascending=False).to_string())

## 5. Feature Engineering

In [ ]:
# Engineered features
df_transformed['UTIL_RATIO']  = df_transformed['BILL_AMT1'] / (df_transformed['LIMIT_BAL'] + 1)
df_transformed['AVG_BILL']    = df_transformed[['BILL_AMT1','BILL_AMT2','BILL_AMT3',
                                                  'BILL_AMT4','BILL_AMT5','BILL_AMT6']].mean(axis=1)
df_transformed['AVG_PAY_AMT'] = df_transformed[['PAY_AMT1','PAY_AMT2','PAY_AMT3',
                                                  'PAY_AMT4','PAY_AMT5','PAY_AMT6']].mean(axis=1)
df_transformed['TOTAL_DELAY'] = df_transformed[['PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6']].sum(axis=1)

print('Engineered features added: UTIL_RATIO, AVG_BILL, AVG_PAY_AMT, TOTAL_DELAY')
df_transformed.shape

## 6. Class Imbalance Treatment — SMOTE

In [ ]:
X = df_transformed.drop('default', axis=1)
y = df_transformed['default']

print(f'Original class distribution:')
print(y.value_counts())
print(f'Imbalance ratio: {y.value_counts()[0]/y.value_counts()[1]:.2f}:1')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Apply SMOTE on training data ONLY
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f'\nAfter SMOTE (train only):')
print(pd.Series(y_train_bal).value_counts())

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#2ecc71', '#e74c3c']
y_train.value_counts().plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Before SMOTE', fontsize=13, fontweight='bold'); axes[0].set_xticklabels(['No Default','Default'], rotation=0)
pd.Series(y_train_bal).value_counts().plot(kind='bar', ax=axes[1], color=colors, edgecolor='black')
axes[1].set_title('After SMOTE', fontsize=13, fontweight='bold'); axes[1].set_xticklabels(['No Default','Default'], rotation=0)
plt.tight_layout()
plt.savefig('models/smote_balance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Model Building & Evaluation

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled  = scaler.transform(X_test)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train_bal)
    preds = model.predict(X_test_scaled)
    proba = model.predict_proba(X_test_scaled)[:, 1]
    results[name] = {
        'accuracy':  round(accuracy_score(y_test, preds), 4),
        'precision': round(precision_score(y_test, preds), 4),
        'recall':    round(recall_score(y_test, preds), 4),
        'f1':        round(f1_score(y_test, preds), 4),
        'roc_auc':   round(roc_auc_score(y_test, proba), 4),
    }
    print(f'{name}: {results[name]}')

results_df = pd.DataFrame(results).T
results_df

In [ ]:
# Confusion Matrix — Random Forest
best_model = models['Random Forest']
preds = best_model.predict(X_test_scaled)
cm = confusion_matrix(y_test, preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'])
plt.title('Confusion Matrix — Random Forest', fontsize=13, fontweight='bold')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('models/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(classification_report(y_test, preds, target_names=['No Default', 'Default']))

In [ ]:
# ROC Curves for all models
plt.figure(figsize=(9, 6))
colors = ['#3498db', '#e67e22', '#2ecc71', '#9b59b6']

for (name, model), color in zip(models.items(), colors):
    proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, lw=2)

plt.plot([0,1],[0,1],'k--', lw=1)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — All Models', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('models/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance
feat_imp = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(12, 6))
feat_imp.head(15).plot(kind='bar', color='#3498db', edgecolor='black')
plt.title('Top 15 Feature Importances — Random Forest', fontsize=13, fontweight='bold')
plt.ylabel('Importance'); plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('models/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Model Comparison Summary

In [ ]:
# Bar chart comparison
metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
results_df[metrics].plot(kind='bar', figsize=(14, 6), edgecolor='black')
plt.title('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Model'); plt.ylabel('Score')
plt.xticks(rotation=15)
plt.legend(loc='lower right')
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig('models/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nBest Model: Random Forest')
print(results_df)

## 9. Save Best Model

In [ ]:
# Save model and scaler — use the ORIGINAL (non-log-transformed) version for app compatibility
df_raw = pd.read_csv('data/Credit_Card_Default.csv')
df_raw.rename(columns={'default.payment.next.month': 'default'}, inplace=True)
df_raw.drop(columns=['ID'], inplace=True)
df_raw['EDUCATION'] = df_raw['EDUCATION'].replace({0:4,5:4,6:4})
df_raw['MARRIAGE']  = df_raw['MARRIAGE'].replace({0:3})

X_raw = df_raw.drop('default', axis=1)
y_raw = df_raw['default']

X_tr, X_te, y_tr, y_te = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw)
X_tr_bal, y_tr_bal = SMOTE(random_state=42).fit_resample(X_tr, y_tr)
sc = StandardScaler()
X_tr_sc = sc.fit_transform(X_tr_bal)

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_tr_sc, y_tr_bal)

joblib.dump(rf, 'models/model.pkl')
joblib.dump(sc, 'models/scaler.pkl')

meta = {'feature_names': X_raw.columns.tolist(), 'model_name': 'Random Forest',
        'metrics': results['Random Forest'], 'all_results': results}
with open('models/meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('✅ model.pkl, scaler.pkl and meta.json saved to models/')